## Modeling

In [9]:
from joblib import dump, load
from pathlib import Path

import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier

In [10]:
RANDOM_STATE = 22
ARTIFACTS_DIR = Path("artifacts")
if not ARTIFACTS_DIR.is_dir():
    ARTIFACTS_DIR = Path("..") / ARTIFACTS_DIR

In [11]:
preprocessing_artifact = load(ARTIFACTS_DIR / "preprocessing.joblib")
preprocessor = preprocessing_artifact["preprocessor"]
target = preprocessing_artifact["target"]

if "train" in preprocessing_artifact:
    X_train = preprocessing_artifact["train"]
else:
    X_train = pd.read_csv(ARTIFACTS_DIR / "X_train.csv", index_col=0)

# Match labels to feature rows using the saved training indices.
y_train = pd.read_csv(ARTIFACTS_DIR / "y_train.csv", index_col=0)[target].reindex(X_train.index)

if y_train.isna().any():
    raise ValueError("Missing labels for the training split. Rerun 02_preprocessing.ipynb.")

In [12]:
X_train.shape, y_train.shape

((1600, 15), (1600,))

In [14]:
param_spaces = {
    "Logistic Regression": {
        "classifier__C": [0.01, 0.1, 1, 10, 100],
        "classifier__class_weight": [None, "balanced"]
    },

    "Random Forest": {
        "classifier__n_estimators": [100, 200, 300],
        "classifier__max_depth": [None, 5, 10, 20],
        "classifier__min_samples_leaf": [1, 2, 4],
        "classifier__class_weight": [None, "balanced"]
    },

    "KNN": {
        "classifier__n_neighbors": [3, 5, 9, 15],
        "classifier__weights": ["uniform", "distance"],
    },
}

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        random_state=RANDOM_STATE,
    ),
    "KNN": KNeighborsClassifier(
        n_neighbors=5,
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        random_state=RANDOM_STATE,
    ),
}

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

scoring = "f1_macro"
results = []
searches = {}
tuning_results = []

In [30]:
def evaluate(models, tuning=False):
    for name, model in models.items():
        pipeline = Pipeline([
            ("preprocessor", preprocessor),
            ("classifier", model),
        ])
        if tuning:
            search = GridSearchCV(
                estimator=pipeline,
                param_grid=param_spaces[name],
                cv=cv,
                scoring=scoring,
                refit=True,
                n_jobs=-1,
                error_score="raise"
            )
            search.fit(X_train,y_train)
            searches[name] = search
            tuning_results.append({
                "Model": name,
                "F1 Macro": search.best_score_,
                "Best Parameters": search.best_params_,
            })
            
        scores = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring=scoring)
        results.append({"Model": name, "F1 Macro": scores.mean(),"std":scores.std()})

    return pd.DataFrame(results).sort_values("F1 Macro", ascending=False ,ignore_index=True) if not tuning else pd.DataFrame(tuning_results).sort_values("F1 Macro", ascending=False, ignore_index=True)

In [ ]:
evaluate(models=models, tuning=False)

,Model,F1 Macro,std
0,Logistic Regression,0.760078,0.014836
1,Random Forest,0.693433,0.023861
2,KNN,0.583378,0.021549


In [ ]:
evaluate(models=models, tuning=True)

In [28]:
best_pipeline = searches["Logistic Regression"].best_estimator_

In [29]:
dump(best_pipeline,  ARTIFACTS_DIR.parent / "models" / "best_pipeline.joblib")

['..\\models\\best_pipeline.joblib']